# 07 – Multi-seed statistical benchmark

Runs the closed-loop benchmark from notebook 02 across **5 random seeds**
to obtain mean ± std performance estimates required for Scopus Q1 review.

Methods compared:
- `rule_based` – deterministic rule-based controller
- `raw_sindy_mpc` – raw-feature SINDy + CasADi MPC
- `physics_sindy_mpc` – physics-informed SINDy + CasADi MPC
- `nn_mpc` – MLP surrogate + shooting MPC (optional, slow)

Results are saved to `results_scenarios/tables/multi_seed_benchmark.csv` and
`multi_seed_summary.csv` (mean ± std).

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / 'article_experiment_utils.py').exists():
    ARTICLE_DIR = HERE
    REPO_DIR = HERE.parent
else:
    ARTICLE_DIR = HERE / 'own-article'
    REPO_DIR = HERE
sys.path.insert(0, str(ARTICLE_DIR))
sys.path.insert(0, str(REPO_DIR))

from article_experiment_utils import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

OUT = results_dir()

In [ ]:
FAST_MODE  = False   # True → 1-day rollout, 3 seeds, no NN (quick smoke-test)
INCLUDE_NN = False   # True → include MLP-MPC (adds ~30 min per seed)

SEEDS         = [42, 123, 456, 789, 1337] if not FAST_MODE else [42, 123, 456]
ROLLOUT_DAYS  = 1 if FAST_MODE else 14
BENCH_DATE    = '2010-04-15'

cfg = ExperimentConfig(n_days=60, start_date='2010-02-28', horizon=20)

# Load pre-collected training data
train_path = OUT / 'datasets' / 'rbc_train_60d.npz'
train_data = load_dataset(train_path) if train_path.exists() \
    else collect_rule_based_dataset(cfg)
save_dataset(train_data, train_path)
train_subset = train_data.subset_steps(60 * cfg.steps_per_day)
print(f'Training rows: {len(train_subset.states)}  |  Seeds: {SEEDS}')

## Run benchmark across seeds

Each seed re-initialises the environment. SINDy models are re-fitted per seed
to capture training variability (though SINDy is deterministic given the same data,
so per-seed fits are identical here — the variance comes from env stochasticity).

Expected time: ~5–15 min total without NN.

In [ ]:
import time
from dataclasses import asdict

all_rows = []

for seed_idx, seed in enumerate(SEEDS):
    print(f'\n── Seed {seed} ({seed_idx+1}/{len(SEEDS)}) ──')
    cfg_s = ExperimentConfig(**{**asdict(cfg), 'seed': seed})

    # Rule-based
    t0 = time.perf_counter()
    df_rbc = rollout_rule_based(
        cfg_s, n_days=ROLLOUT_DAYS, start_date=BENCH_DATE, noise_scale=0.0, seed=seed
    )
    m = rollout_metrics(df_rbc)
    m.update({'method': 'rule_based', 'seed': seed, 'wall_time_s': round(time.perf_counter()-t0, 2)})
    all_rows.append(m)
    print(f'  rule_based  | temp_rmse={m["temp_rmse"]:.3f}  comfort={m["comfort_pct"]:.1f}%')

    # Raw SINDy-MPC
    bundle_raw = fit_sindy(
        train_subset, feature_variant='raw', library_degree=1,
        period=float(cfg.period), metadata={'label': f'raw_sindy_s{seed}'}
    )
    t0 = time.perf_counter()
    df_raw = rollout_mpc(
        bundle_raw, cfg_s, n_days=ROLLOUT_DAYS, start_date=BENCH_DATE, objective='full'
    )
    m = rollout_metrics(df_raw)
    m.update({'method': 'raw_sindy_mpc', 'seed': seed, 'wall_time_s': round(time.perf_counter()-t0, 2)})
    all_rows.append(m)
    print(f'  raw_sindy   | temp_rmse={m["temp_rmse"]:.3f}  comfort={m["comfort_pct"]:.1f}%')

    # Physics SINDy-MPC
    bundle_pi = fit_sindy(
        train_subset, feature_variant='physics', library_degree=1,
        period=float(cfg.period), metadata={'label': f'physics_sindy_s{seed}'}
    )
    t0 = time.perf_counter()
    df_pi = rollout_mpc(
        bundle_pi, cfg_s, n_days=ROLLOUT_DAYS, start_date=BENCH_DATE, objective='full'
    )
    m = rollout_metrics(df_pi)
    m.update({'method': 'physics_sindy_mpc', 'seed': seed, 'wall_time_s': round(time.perf_counter()-t0, 2)})
    all_rows.append(m)
    print(f'  physics_si  | temp_rmse={m["temp_rmse"]:.3f}  comfort={m["comfort_pct"]:.1f}%')

    # DAgger model (if available)
    dagger_path = OUT / 'models' / 'dagger_iter_3.pkl'
    if dagger_path.exists():
        bundle_dag = load_bundle(dagger_path)
        bundle_dag.metadata['label'] = f'physics_sindy_dagger_s{seed}'
        t0 = time.perf_counter()
        df_dag = rollout_mpc(
            bundle_dag, cfg_s, n_days=ROLLOUT_DAYS, start_date=BENCH_DATE, objective='full'
        )
        m = rollout_metrics(df_dag)
        m.update({'method': 'physics_sindy_dagger', 'seed': seed, 'wall_time_s': round(time.perf_counter()-t0, 2)})
        all_rows.append(m)
        print(f'  phys+dagger | temp_rmse={m["temp_rmse"]:.3f}  comfort={m["comfort_pct"]:.1f}%')

    # NN-MPC (optional, slow)
    if INCLUDE_NN:
        nn_path = OUT / 'models' / 'nn_mpc_physics.pkl'
        nn_b = load_bundle(nn_path) if nn_path.exists() else fit_nn_surrogate(
            train_subset, feature_variant='physics', hidden_sizes=[64, 64],
            epochs=300, period=float(cfg.period), metadata={'label': f'nn_s{seed}'}
        )
        t0 = time.perf_counter()
        df_nn = rollout_mpc_nn(
            nn_b, cfg_s, n_days=ROLLOUT_DAYS, start_date=BENCH_DATE, horizon=cfg.horizon
        )
        m = rollout_metrics(df_nn)
        m.update({'method': 'nn_mpc', 'seed': seed, 'wall_time_s': round(time.perf_counter()-t0, 2)})
        all_rows.append(m)
        print(f'  nn_mpc      | temp_rmse={m["temp_rmse"]:.3f}  comfort={m["comfort_pct"]:.1f}%')

seed_df = pd.DataFrame(all_rows)
save_table(seed_df, OUT / 'tables' / 'multi_seed_benchmark.csv')
print(f'\nSaved {len(seed_df)} rows to multi_seed_benchmark.csv')

## Statistical summary (mean ± std)

In [ ]:
key_metrics = ['temp_rmse', 'co2_rmse', 'comfort_pct', 'energy_proxy_sum',
               'temp_low_violations', 'rh_excess_area', 'wall_time_s']
available = [c for c in key_metrics if c in seed_df.columns]

mean_df = seed_df.groupby('method')[available].mean().round(3)
std_df  = seed_df.groupby('method')[available].std().round(3)

# Build paper-ready table: "mean ± std" strings
summary_rows = []
for method in mean_df.index:
    row = {'method': method}
    for col in available:
        row[col] = f"{mean_df.loc[method, col]:.3f} ± {std_df.loc[method, col]:.3f}"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
save_table(summary, OUT / 'tables' / 'multi_seed_summary.csv')
summary

## Statistical significance test (Wilcoxon signed-rank)

Tests whether `physics_sindy_mpc` is significantly better than `rule_based` and `raw_sindy_mpc`.

In [ ]:
from scipy import stats

def wilcoxon_pair(df, method_a, method_b, metric, alternative='two-sided'):
    """Wilcoxon signed-rank test on per-seed metric values."""
    a = df[df['method'] == method_a].set_index('seed')[metric]
    b = df[df['method'] == method_b].set_index('seed')[metric]
    common = a.index.intersection(b.index)
    if len(common) < 3:
        return {'method_a': method_a, 'method_b': method_b, 'metric': metric,
                'p_value': float('nan'), 'significant_p05': False, 'note': 'too few seeds'}
    try:
        stat, p = stats.wilcoxon(a[common].values, b[common].values, alternative=alternative)
    except ValueError as e:
        stat, p = float('nan'), float('nan')
    return {
        'method_a': method_a, 'method_b': method_b, 'metric': metric,
        'statistic': float(stat), 'p_value': float(p),
        'significant_p05': bool(p < 0.05),
        'mean_a': float(a[common].mean()), 'mean_b': float(b[common].mean()),
    }

sig_rows = []
pairs = [
    ('physics_sindy_mpc', 'rule_based', 'temp_rmse', 'less'),
    ('physics_sindy_mpc', 'raw_sindy_mpc', 'temp_rmse', 'two-sided'),
    ('physics_sindy_mpc', 'rule_based', 'energy_proxy_sum', 'less'),
    ('physics_sindy_mpc', 'rule_based', 'comfort_pct', 'greater'),
]
if 'physics_sindy_dagger' in seed_df['method'].values:
    pairs += [
        ('physics_sindy_dagger', 'physics_sindy_mpc', 'temp_rmse', 'two-sided'),
    ]
if INCLUDE_NN and 'nn_mpc' in seed_df['method'].values:
    pairs += [
        ('physics_sindy_mpc', 'nn_mpc', 'temp_rmse', 'two-sided'),
    ]

for a, b, metric, alt in pairs:
    if a in seed_df['method'].values and b in seed_df['method'].values:
        sig_rows.append(wilcoxon_pair(seed_df, a, b, metric, alternative=alt))

sig_df = pd.DataFrame(sig_rows)
save_table(sig_df, OUT / 'tables' / 'significance_tests.csv')
sig_df[['method_a', 'method_b', 'metric', 'mean_a', 'mean_b', 'p_value', 'significant_p05']].round(4)

## Visualisation: metric distributions across seeds

In [ ]:
plot_metrics = [
    ('temp_rmse', 'Temperature RMSE (°C)'),
    ('co2_rmse', 'CO₂ RMSE (ppm)'),
    ('comfort_pct', 'Comfort zone (%)'),
    ('energy_proxy_sum', 'Energy proxy (sum)'),
]

available_plot = [(m, l) for m, l in plot_metrics if m in seed_df.columns]
methods = seed_df['method'].unique()
colors  = plt.cm.tab10(np.linspace(0, 0.8, len(methods)))
method_colors = dict(zip(methods, colors))

fig, axes = plt.subplots(1, len(available_plot), figsize=(4 * len(available_plot), 4))
if len(available_plot) == 1:
    axes = [axes]

for ax, (metric, label) in zip(axes, available_plot):
    data_by_method = [seed_df[seed_df['method'] == m][metric].values for m in methods]
    bp = ax.boxplot(data_by_method, labels=methods, patch_artist=True)
    for patch, method in zip(bp['boxes'], methods):
        patch.set_facecolor(method_colors[method])
        patch.set_alpha(0.7)
    ax.set_ylabel(label)
    ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle(f'Performance across {len(SEEDS)} seeds  (14-day rollout)', fontsize=11)
fig.tight_layout()
save_figure(fig, OUT / 'figures' / 'multi_seed_boxplot.png')
plt.show()

In [ ]:
# Per-seed line plot for temp_rmse – shows consistency across runs
if 'temp_rmse' in seed_df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    for method in methods:
        grp = seed_df[seed_df['method'] == method].sort_values('seed')
        ax.plot(grp['seed'], grp['temp_rmse'], marker='o',
                label=method, color=method_colors[method])
    ax.set_xlabel('Random seed')
    ax.set_ylabel('Temperature RMSE (°C)')
    ax.set_title('Per-seed temperature RMSE')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_figure(fig, OUT / 'figures' / 'multi_seed_per_seed_temp_rmse.png')
    plt.show()